# 第二部分：数据清洗

本notebook完成10只股票的原始数据清洗、宽表长表转换、多表合并任务。

**股票列表**：招商银行(600036)、工商银行(601398)、赛力斯(601127)、长安汽车(000625)、保利发展(600048)、招商蛇口(001979)、贵州茅台(600519)、泸州老窖(000568)、隆基绿能(601012)、阳光电源(300274)

**指数数据**：沪深300指数(000300)

**宏观数据**：CPI同比数据

## 3.1 单表清洗

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

WORK_DIR = r"D:\02 个人资料\中大 2025MDE\2026 研一 第二学期\数据分析与经济决策\P01_25210291_曾垂健\dshw-p01"
os.chdir(WORK_DIR)
os.makedirs('data/clean', exist_ok=True)
os.makedirs('data/combined', exist_ok=True)

stock_codes = ['600036', '601398', '601127', '000625', '600048', '001979', '600519', '000568', '601012', '300274']
stock_names = {'600036': '招商银行', '601398': '工商银行', '601127': '赛力斯', '000625': '长安汽车', 
                '600048': '保利发展', '001979': '招商蛇口', '600519': '贵州茅台', '000568': '泸州老窖', 
                '601012': '隆基绿能', '300274': '阳光电源'}

### 3.1.1 读取数据并检测缺失值

In [ ]:
# 读取所有股票数据并检测缺失值
missing_stats = []

for code in stock_codes:
    df = pd.read_csv(f'data/stock/{code}.csv', encoding='utf-8-sig')
    print(f"\n=== {stock_names[code]}({code}) ===")
    print(f"原始列名: {df.columns.tolist()}")
    print(f"原始行数: {len(df)}")
    
    # 统计缺失值
    price_cols = ['开盘', '收盘', '最高', '最低', '成交量']
    if '成交额' in df.columns:
        price_cols.append('成交额')
    missing = df[price_cols].isnull().sum()
    missing_total = missing.sum()
    print(f"缺失值总数: {missing_total}")
    if missing_total > 0:
        for col, cnt in missing.items():
            if cnt > 0:
                missing_stats.append({'stock': code, 'column': col, 'null_count': cnt})

**缺失值分析**：A股日度行情数据缺失原因可能是：
- 股票停牌日（无交易）
- 数据接口返回null
- 部分股票数据缺少成交额字段

### 3.1.2 数据清洗处理

In [ ]:
# 对每只股票进行清洗
all_clean = []

for code in stock_codes:
    df = pd.read_csv(f'data/stock/{code}.csv', encoding='utf-8-sig')
    
    # 1. 日期格式统一为datetime64
    df['date'] = pd.to_datetime(df['日期'])
    
    # 2. 数据类型检查与转换（价格、成交量为数值型）
    price_cols = ['开盘', '收盘', '最高', '最低', '成交量']
    for col in price_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 成交额可能不存在，不存在时填充NaN
    if '成交额' in df.columns:
        df['成交额'] = pd.to_numeric(df['成交额'], errors='coerce')
    else:
        df['成交额'] = np.nan
    
    # 3. 缺失值处理：向前填充(ffill)
    check_cols = ['开盘', '收盘', '最高', '最低', '成交量', '成交额']
    df[check_cols] = df[check_cols].ffill()
    
    # 4. 重复值检测与删除
    dup_count = df.duplicated(subset=['date']).sum()
    df = df.drop_duplicates(subset=['date'])
    print(f"{stock_names[code]}({code}): 删除重复行 {dup_count}")
    
    # 5. 计算日收益率 daily_return = np.log(close / close.shift(1)) * 100
    df['daily_return'] = np.log(df['收盘'] / df['收盘'].shift(1)) * 100
    
    # 6. 标注离群值 is_extreme = abs(daily_return) > 20
    df['is_extreme'] = abs(df['daily_return']) > 20
    extreme_count = df['is_extreme'].sum()
    print(f"  离群值(|daily_return|>20): {extreme_count}")
    
    # 7. 选择需要的列并重命名为英文
    df_clean = df[['date', '开盘', '收盘', '最高', '最低', '成交量', '成交额', 'daily_return', 'is_extreme']].copy()
    df_clean['code'] = code
    df_clean = df_clean.rename(columns={'开盘': 'open', '收盘': 'close', '最高': 'high', '最低': 'low', '成交量': 'volume', '成交额': 'amount'})
    df_clean = df_clean[['date', 'code', 'open', 'high', 'low', 'close', 'volume', 'amount', 'daily_return', 'is_extreme']]
    all_clean.append(df_clean)

print(f"\n共清洗 {len(all_clean)} 只股票")

**缺失值处理选择依据**：
- 价格数据（开盘、收盘、最高、最低）采用向前填充(ffill)，因为相邻交易日的价格具有连续性
- 成交量等数据也采用ffill，保持数据一致性
- 部分股票缺少成交额字段，统一填充NaN

## 3.2 合并所有股票数据为长表

In [ ]:
# 合并为长表
df_clean_all = pd.concat(all_clean, ignore_index=True)
df_clean_all = df_clean_all.sort_values(['code', 'date']).reset_index(drop=True)
print(f"清洗后总行数: {len(df_clean_all)}")
print(f"股票数量: {df_clean_all['code'].nunique()}")
print(f"日期范围: {df_clean_all['date'].min()} ~ {df_clean_all['date'].max()}")
print(f"\n字段: {df_clean_all.columns.tolist()}")
print(df_clean_all.head())

### 保存清洗后数据

In [ ]:
# 保存CSV和Parquet格式
df_clean_all.to_csv('data/clean/stock_clean.csv', index=False)
df_clean_all.to_parquet('data/clean/stock_clean.parquet', index=False)
print("已保存 stock_clean.csv 和 stock_clean.parquet")

## 3.3 宽表与长表转换

In [ ]:
# 将10只股票的收盘价合并为宽表（日期为索引，每列一只股票）
pivot_close = df_clean_all.pivot_table(index='date', columns='code', values='close', aggfunc='first')
print(f"宽表形状: {pivot_close.shape}")
print("\n宽表前5行：")
print(pivot_close.head())

In [ ]:
# 用pd.melt将宽表转换回长表
long_close = pivot_close.reset_index().melt(id_vars='date', var_name='code', value_name='close')
print(f"长表形状: {long_close.shape}")
print("\n长表前10行：")
print(long_close.head(10))

**宽表与长表适用场景**：

- **宽表适合**：
  - 多个时间序列的横向对比（如不同股票价格走势）
  - 计算相关系数、协方差矩阵
  - 数据透视、交叉表分析
  - 便于绘制多线图

- **长表适合**：
  - 分组操作（groupby）
  - 存储到数据库
  - 纵向合并、关联操作
  - 机器学习特征工程

## 3.4 多表合并

### 3.4.1 读取并清洗指数数据

In [ ]:
# 读取沪深300指数数据
df_index = pd.read_csv('data/index/index_000300.csv')
print(f"指数列名: {df_index.columns.tolist()}")
print(f"指数原始行数: {len(df_index)}")

df_index['date'] = pd.to_datetime(df_index['date'])
df_index['index_close'] = pd.to_numeric(df_index['close'], errors='coerce')
df_index['index_return'] = np.log(df_index['index_close'] / df_index['index_close'].shift(1)) * 100
df_index = df_index[['date', 'index_close', 'index_return']].dropna()
print(f"指数清洗后行数: {len(df_index)}")
print(df_index.head())

### 3.4.2 个股数据与指数数据合并（left join）

In [ ]:
# 个股日度数据与指数日度数据按日期做left join
print("\n=== Step 1: 个股 + 指数合并 ===")
rows_before = len(df_clean_all)
print(f"合并前行数: {rows_before}")

df_combined = df_clean_all.merge(df_index, on='date', how='left')
print(f"合并后行数: {len(df_combined)}")
print(f"index_close匹配率: {df_combined['index_close'].notna().sum()}/{len(df_combined)}")

### 3.4.3 读取并清洗CPI宏观数据

In [ ]:
# 读取CPI数据
df_cpi = pd.read_csv('data/macro/macro_cpi.csv')
print(f"CPI列名: {df_cpi.columns.tolist()}")
print(f"CPI原始行数: {len(df_cpi)}")
print(df_cpi.head())

In [ ]:
# 清洗CPI数据
df_cpi['month'] = pd.to_datetime(df_cpi['month'])
df_cpi['cpi_yoy'] = pd.to_numeric(df_cpi['cpi_yoy'], errors='coerce')
df_cpi['year_month'] = df_cpi['month'].dt.to_period('M')
df_cpi = df_cpi[['year_month', 'cpi_yoy']].drop_duplicates()
print(f"CPI清洗后行数: {len(df_cpi)}")
print(f"CPI日期范围: {df_cpi['month'].min()} ~ {df_cpi['month'].max()}")
print(df_cpi.head())

### 3.4.4 宏观数据映射到日度数据

In [ ]:
# 将月度CPI数据映射到对应月份的每个交易日
print("\n=== Step 2: 日度 + 月度宏观合并 ===")
rows_before2 = len(df_combined)
print(f"合并前行数: {rows_before2}")

# 为日度数据添加月份列用于匹配
df_combined['year_month'] = pd.to_datetime(df_combined['date']).dt.to_period('M')

# 合并CPI数据
df_combined = df_combined.merge(df_cpi, on='year_month', how='left')
print(f"合并后行数: {len(df_combined)}")
print(f"cpi_yoy匹配率: {df_combined['cpi_yoy'].notna().sum()}/{len(df_combined)}")

## 3.5 保存综合数据

In [ ]:
# 保存综合数据
df_combined.to_csv('data/combined/combined_data.csv', index=False)
print("已保存 combined_data.csv")

## 清洗效果统计

In [ ]:
print("=" * 50)
print("数据清洗完成")
print("=" * 50)
print(f"股票数量: {df_combined['code'].nunique()}")
print(f"日期范围: {df_combined['date'].min()} ~ {df_combined['date'].max()}")
print(f"总行数: {len(df_combined)}")
print(f"缺失值总数: {df_combined.isnull().sum().sum()}")
print(f"离群值总数(|daily_return|>20): {df_combined['is_extreme'].sum()}")

print("\n各股票数据量：")
print(df_combined.groupby('code').size())

print("\n数据类型：")
print(df_combined.dtypes)

In [ ]:
# 显示前几行数据
print("清洗后数据预览：")
print(df_combined.head(10))

## 输出文件清单

| 文件 | 路径 | 说明 |
|------|------|------|
| stock_clean.csv | data/clean/ | 清洗后长表数据(CSV) |
| stock_clean.parquet | data/clean/ | 清洗后长表数据(Parquet) |
| combined_data.csv | data/combined/ | 合并后综合数据(含指数和CPI) |